# 10 · Window Embedding and Clustering

This notebook turns local gap windows into **descriptor vectors** and studies their geometry in a low-dimensional embedding.

We compare three ensembles:

1. zeta-zero gaps
2. Poisson / exponential control gaps
3. finite GUE-matrix bulk gaps

The goal is to see whether local-window descriptors for zeta and GUE occupy similar regions of feature space, and whether Poisson separates more clearly.

## Why this matters

Earlier notebooks showed that zeta and GUE behave similarly in:
- spacing statistics
- ratio statistics
- pair-correlation behavior
- higher-order 4-gap / 5-gap descriptors
- conditional local responses

This notebook asks a more geometric question:

> if each local window is represented by a feature vector, how do the ensembles cluster?

The workflow is:

1. extract 5-gap windows
2. normalize each window by its mean
3. compute descriptor vectors
4. reduce to 2D with PCA
5. compare ensemble clouds and simple cluster structure

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Data sources

In [ ]:
# Zeta gaps
N = 700
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps = np.diff(t)

# Poisson control
poisson_gaps = rng.exponential(scale=1.0, size=len(zeta_gaps))

# Finite GUE bulk gaps
def sample_gue_bulk_spacings(matrix_size=140, n_matrices=40, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep

        bulk_vals = eigvals[start:stop]
        bulk_gaps = np.diff(bulk_vals)
        all_gaps.extend(bulk_gaps.tolist())

    return np.array(all_gaps)

gue_gaps = sample_gue_bulk_spacings(matrix_size=140, n_matrices=40, bulk_fraction=0.5, rng=rng)

len(zeta_gaps), len(poisson_gaps), len(gue_gaps)

## Build 5-gap windows and normalize each by its mean

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

zeta_w = extract_windows(zeta_gaps, k=5)
poisson_w = extract_windows(poisson_gaps, k=5)
gue_w = extract_windows(gue_gaps, k=5)

zeta_wn = np.array([normalize_window(w) for w in zeta_w])
poisson_wn = np.array([normalize_window(w) for w in poisson_w])
gue_wn = np.array([normalize_window(w) for w in gue_w])

zeta_wn.shape, gue_wn.shape

## Descriptor functions

Each window becomes a feature vector containing:

- the 5 normalized coordinates
- unevenness
- reversal symmetry
- entropy
- mean consecutive ratio
- std of consecutive ratios

In [ ]:
def unevenness(window):
    return float(np.max(window) - np.min(window))

def reversal_symmetry_score(window):
    return float(np.mean(np.abs(window - window[::-1])))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_features(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r)), float(np.std(r))

def descriptor_vector(window):
    ratio_mean, ratio_std = ratio_features(window)
    return np.array([
        window[0], window[1], window[2], window[3], window[4],
        unevenness(window),
        reversal_symmetry_score(window),
        window_entropy(window),
        ratio_mean,
        ratio_std,
    ], dtype=float)

zeta_desc = np.array([descriptor_vector(w) for w in zeta_wn])
poisson_desc = np.array([descriptor_vector(w) for w in poisson_wn])
gue_desc = np.array([descriptor_vector(w) for w in gue_wn])

zeta_desc.shape, poisson_desc.shape, gue_desc.shape

## Standardize the descriptor space

In [ ]:
all_desc = np.vstack([zeta_desc, poisson_desc, gue_desc])

mean = all_desc.mean(axis=0)
std = all_desc.std(axis=0)
std[std == 0] = 1.0

zeta_std = (zeta_desc - mean) / std
poisson_std = (poisson_desc - mean) / std
gue_std = (gue_desc - mean) / std

## PCA embedding

We implement PCA directly with NumPy using the covariance matrix and eigenvectors.

In [ ]:
X = np.vstack([zeta_std, poisson_std, gue_std])
X_centered = X - X.mean(axis=0)

cov = np.cov(X_centered, rowvar=False)
eigvals, eigvecs = np.linalg.eigh(cov)

order = np.argsort(eigvals)[::-1]
eigvals = eigvals[order]
eigvecs = eigvecs[:, order]

components = eigvecs[:, :2]
embedding = X_centered @ components

n_zeta = len(zeta_std)
n_poisson = len(poisson_std)
n_gue = len(gue_std)

zeta_emb = embedding[:n_zeta]
poisson_emb = embedding[n_zeta:n_zeta + n_poisson]
gue_emb = embedding[n_zeta + n_poisson:]

explained = eigvals[:2] / eigvals.sum()
explained

## PCA scatter plot by ensemble

In [ ]:
plt.figure(figsize=(8.8, 6.2))
plt.scatter(zeta_emb[:, 0], zeta_emb[:, 1], s=10, alpha=0.35, label="zeta")
plt.scatter(poisson_emb[:, 0], poisson_emb[:, 1], s=10, alpha=0.35, label="Poisson")
plt.scatter(gue_emb[:, 0], gue_emb[:, 1], s=10, alpha=0.35, label="GUE")
plt.xlabel(f"PC1 ({explained[0]*100:.1f}% var)")
plt.ylabel(f"PC2 ({explained[1]*100:.1f}% var)")
plt.title("PCA embedding of 5-gap descriptor vectors")
plt.legend()
plt.show()

## Ensemble centroids in PCA space

In [ ]:
centroids = {
    "zeta": zeta_emb.mean(axis=0),
    "Poisson": poisson_emb.mean(axis=0),
    "GUE": gue_emb.mean(axis=0),
}
centroids

In [ ]:
plt.figure(figsize=(8.0, 6.0))
plt.scatter(zeta_emb[:, 0], zeta_emb[:, 1], s=8, alpha=0.20, label="zeta")
plt.scatter(poisson_emb[:, 0], poisson_emb[:, 1], s=8, alpha=0.20, label="Poisson")
plt.scatter(gue_emb[:, 0], gue_emb[:, 1], s=8, alpha=0.20, label="GUE")

for name, c in centroids.items():
    plt.scatter(c[0], c[1], s=160, marker="X", label=f"{name} centroid")

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA embedding with ensemble centroids")
plt.legend()
plt.show()

## Simple distance-to-centroid summary

This gives a compact geometric comparison.

In [ ]:
def mean_distance_to_point(X, point):
    return float(np.mean(np.linalg.norm(X - point, axis=1)))

distance_summary = {
    "distance(zeta, zeta_centroid)": mean_distance_to_point(zeta_emb, centroids["zeta"]),
    "distance(Poisson, Poisson_centroid)": mean_distance_to_point(poisson_emb, centroids["Poisson"]),
    "distance(GUE, GUE_centroid)": mean_distance_to_point(gue_emb, centroids["GUE"]),
    "distance(zeta_centroid, GUE_centroid)": float(np.linalg.norm(centroids["zeta"] - centroids["GUE"])),
    "distance(zeta_centroid, Poisson_centroid)": float(np.linalg.norm(centroids["zeta"] - centroids["Poisson"])),
    "distance(GUE_centroid, Poisson_centroid)": float(np.linalg.norm(centroids["GUE"] - centroids["Poisson"])),
}
distance_summary

## Simple k-means clustering (k=3)

We implement a small k-means routine directly with NumPy.
This is exploratory: the goal is not optimal clustering, but to see whether descriptor space separates the ensembles in a simple way.

In [ ]:
def kmeans(X, k=3, n_iter=30, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    idx = rng.choice(len(X), size=k, replace=False)
    centers = X[idx].copy()

    for _ in range(n_iter):
        dists = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2)
        labels = np.argmin(dists, axis=1)

        new_centers = []
        for j in range(k):
            cluster = X[labels == j]
            if len(cluster) == 0:
                new_centers.append(centers[j])
            else:
                new_centers.append(cluster.mean(axis=0))
        new_centers = np.array(new_centers)

        if np.allclose(new_centers, centers):
            break
        centers = new_centers

    return labels, centers

labels, km_centers = kmeans(embedding, k=3, n_iter=50, rng=rng)

In [ ]:
plt.figure(figsize=(8.8, 6.2))
plt.scatter(embedding[:, 0], embedding[:, 1], c=labels, s=10, alpha=0.35)
plt.scatter(km_centers[:, 0], km_centers[:, 1], s=180, marker="X")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("K-means clusters in PCA space")
plt.show()

## Cross-tab: ensemble membership vs cluster label

In [ ]:
ensemble_labels = (
    ["zeta"] * n_zeta +
    ["Poisson"] * n_poisson +
    ["GUE"] * n_gue
)

cluster_table = {}
for ensemble_name in ["zeta", "Poisson", "GUE"]:
    mask = np.array([e == ensemble_name for e in ensemble_labels])
    counts = np.bincount(labels[mask], minlength=3)
    cluster_table[ensemble_name] = counts.tolist()

cluster_table

## Conditional overlay: after small vs after large gaps (zeta only)

We reuse the conditional-window idea and color zeta windows according to whether the first gap in the window is from the bottom or top 20% of zeta's normalized gap sequence.

In [ ]:
zeta_g_norm = zeta_gaps / zeta_gaps.mean()
low_thr, high_thr = np.quantile(zeta_g_norm, 0.2), np.quantile(zeta_g_norm, 0.8)

zeta_small_mask = []
zeta_large_mask = []
for i in range(len(zeta_g_norm) - 5 + 1):
    zeta_small_mask.append(zeta_g_norm[i] <= low_thr)
    zeta_large_mask.append(zeta_g_norm[i] >= high_thr)

zeta_small_mask = np.array(zeta_small_mask)
zeta_large_mask = np.array(zeta_large_mask)

zeta_small_emb = zeta_emb[zeta_small_mask]
zeta_large_emb = zeta_emb[zeta_large_mask]

zeta_small_emb.shape, zeta_large_emb.shape

In [ ]:
plt.figure(figsize=(8.8, 6.2))
plt.scatter(zeta_emb[:, 0], zeta_emb[:, 1], s=8, alpha=0.10, label="all zeta windows")
plt.scatter(zeta_small_emb[:, 0], zeta_small_emb[:, 1], s=18, alpha=0.45, label="zeta after small")
plt.scatter(zeta_large_emb[:, 0], zeta_large_emb[:, 1], s=18, alpha=0.45, label="zeta after large")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Conditional zeta windows in PCA space")
plt.legend()
plt.show()

## PCA loading vectors

These tell us which descriptor directions dominate PC1 and PC2.

In [ ]:
feature_names = [
    "w1", "w2", "w3", "w4", "w5",
    "unevenness", "reversal_symmetry", "entropy", "ratio_mean", "ratio_std"
]

loadings = {
    "PC1": {name: float(val) for name, val in zip(feature_names, components[:, 0])},
    "PC2": {name: float(val) for name, val in zip(feature_names, components[:, 1])},
}
loadings

## Notes

- If zeta and GUE clouds overlap more than either overlaps Poisson, that supports the idea that local higher-order structure is shared by zeta zeros and random Hermitian spectra.
- The embedding is exploratory and low-dimensional; overlap or separation should be interpreted qualitatively unless followed by a formal classifier or distance test.
- The conditional zeta overlay helps show whether "after small gap" and "after large gap" windows occupy distinct regions of the same descriptor space.

## Next directions

A natural next notebook could do one of these:

1. add a formal classifier and cross-validation score for zeta / Poisson / GUE windows  
2. compare several descriptor sets and PCA variants  
3. test conditional embeddings for GUE and Poisson as well as zeta  
4. combine this with bootstrap significance on centroid distances